# 06 — Inversas de Matrices

**Objetivo:** Entender qué es la inversa, cuándo existe, cómo calcularla (Gauss-Jordan), cuándo NO usarla, y la pseudoinversa para casos no cuadrados.

## Contexto matemático

La **inversa** de $A \in \mathbb{R}^{n\times n}$ es la única matriz $A^{-1}$ tal que:

$$A A^{-1} = A^{-1} A = I$$

**Existe si y solo si:** $A$ es cuadrada y $\det(A) \neq 0$.

**Uso analítico:** Si $A\mathbf{x} = \mathbf{b}$, entonces $\mathbf{x} = A^{-1}\mathbf{b}$. Sin embargo, en la práctica usamos `np.linalg.solve` (más estable).

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)
print('Setup listo')

## 1 — Gauss-Jordan: calcular la inversa paso a paso

Aumentamos $[A \mid I]$ y aplicamos operaciones de fila hasta llegar a $[I \mid A^{-1}]$.

In [ ]:
def gauss_jordan_inv(A):
    """Calcula A^{-1} via eliminación Gauss-Jordan."""
    n = A.shape[0]
    Aug = np.hstack([A.astype(float).copy(), np.eye(n)])

    for col in range(n):
        # Pivoteo parcial
        max_row = np.argmax(np.abs(Aug[col:, col])) + col
        Aug[[col, max_row]] = Aug[[max_row, col]]

        # Escalar fila pivote
        Aug[col] = Aug[col] / Aug[col, col]

        # Eliminar en TODAS las filas (no solo debajo)
        for row in range(n):
            if row != col:
                Aug[row] -= Aug[row, col] * Aug[col]

    return Aug[:, n:]

# Matriz de atribución 3×3
A_attr = np.array([
    [0.40, 0.25, 0.15],
    [0.30, 0.50, 0.20],
    [0.30, 0.25, 0.65],
])

A_inv_manual = gauss_jordan_inv(A_attr)
A_inv_numpy  = np.linalg.inv(A_attr)

print('A^{-1} (Gauss-Jordan):')
print(A_inv_manual.round(4))
print('\n¿Coincide con np.linalg.inv?', np.allclose(A_inv_manual, A_inv_numpy))
print('\nVerificación A @ A^{-1} = I:')
print((A_attr @ A_inv_numpy).round(6))

## 2 — `np.linalg.inv` vs `np.linalg.solve`

> **Regla de oro:** nunca uses `inv(A) @ b` para resolver $A\mathbf{x}=\mathbf{b}$. Usa siempre `np.linalg.solve(A, b)`.

¿Por qué? `solve` usa factorización LU con pivoteo parcial directamente sin materializar $A^{-1}$. Es más rápido y numéricamente más estable, especialmente cuando $A$ está mal condicionada.

In [ ]:
import time

# Comparar estabilidad numérica con matriz mal condicionada
np.random.seed(0)
n = 200
A_big = np.random.randn(n, n)
# Hacer A mal condicionada escalando algunas filas
A_big[0] *= 1e-6
b_big = np.random.randn(n)

# Método 1: inv @ b
t0 = time.perf_counter()
x_inv = np.linalg.inv(A_big) @ b_big
t_inv = time.perf_counter() - t0

# Método 2: solve
t0 = time.perf_counter()
x_sol = np.linalg.solve(A_big, b_big)
t_sol = time.perf_counter() - t0

residuo_inv = np.linalg.norm(A_big @ x_inv - b_big)
residuo_sol = np.linalg.norm(A_big @ x_sol - b_big)

print(f'Residuo ||Ax-b|| con inv@b : {residuo_inv:.2e}')
print(f'Residuo ||Ax-b|| con solve  : {residuo_sol:.2e}')
print(f'\nTiempo inv@b : {t_inv*1000:.2f} ms')
print(f'Tiempo solve : {t_sol*1000:.2f} ms')
print(f'\nnp.linalg.cond(A) = {np.linalg.cond(A_big):.2e}')

## 3 — Pseudoinversa de Moore-Penrose

Para matrices **no cuadradas** o **singulares**, la inversa no existe. La **pseudoinversa** $A^+$ generaliza la inversa:

$$A^+ = \lim_{\delta\to 0}(A^T A + \delta I)^{-1}A^T$$

En la práctica se calcula via SVD: $A = U\Sigma V^T \Rightarrow A^+ = V\Sigma^+ U^T$.

- Si $A$ es cuadrada e invertible: $A^+ = A^{-1}$.
- Si $A$ tiene más filas que columnas (overdetermined, $m>n$): $A^+\mathbf{b}$ da la solución de mínimos cuadrados.
- Si $A$ tiene más columnas que filas (underdetermined, $m<n$): $A^+\mathbf{b}$ da la solución de norma mínima.

In [ ]:
# Atribución overdetermined: 6 períodos, 3 canales
# Queremos estimar la contribución base de cada canal
np.random.seed(42)
A_over = np.array([
    [1200., 800., 500.],
    [ 900., 1100., 700.],
    [ 600.,  950., 1300.],
    [1400., 750., 600.],
    [ 800., 1200., 900.],
    [1100., 900., 1100.],
])
b_over = np.array([148., 162., 201., 155., 175., 182.])

# Pseudoinversa → solución de mínimos cuadrados
A_pinv = np.linalg.pinv(A_over)
x_pinv = A_pinv @ b_over

print(f'Shape A: {A_over.shape}  → sistema overdetermined (más ecuaciones que incógnitas)')
print(f'Tasas estimadas (mínimos cuadrados):')
for canal, tasa in zip(['Organic', 'Paid', 'Email'], x_pinv):
    print(f'  {canal}: {tasa*100:.4f}%')

residuo = np.linalg.norm(A_over @ x_pinv - b_over)
print(f'\nResiduo ||Ax - b|| = {residuo:.4f}  (mínimo posible para sistema overdetermined)')

## 4 — Número de condición

El **número de condición** $\kappa(A) = \|A\| \cdot \|A^{-1}\|$ mide cuánto amplifica $A$ los errores de redondeo.

- $\kappa \approx 1$: bien condicionada, solución muy estable.
- $\kappa \approx 10^k$: se pierden $k$ dígitos de precisión.
- $\kappa > 10^{12}$: prácticamente singular en double precision.

In [ ]:
matrices = {
    'Identidad (κ=1)': np.eye(4),
    'Bien condicionada': np.array([[4,1,0,0],[1,4,1,0],[0,1,4,1],[0,0,1,4]], dtype=float),
    'Hilbert 4×4 (κ>>1)': np.array([[1/(i+j-1) for j in range(1,5)] for i in range(1,5)]),
}

print(f'{"Matriz":<30} {"κ (cond)":>15} {"log10(κ)":>10}')
print('-'*55)
for nombre, M in matrices.items():
    kappa = np.linalg.cond(M)
    print(f'{nombre:<30} {kappa:>15.2e} {np.log10(kappa):>10.1f}')

## Resumen

| Concepto | Descripción | NumPy |
|---|---|---|
| Inversa | $AA^{-1}=I$, existe si det≠0 | `np.linalg.inv(A)` |
| Resolver $A\mathbf{x}=\mathbf{b}$ | Más estable que `inv(A)@b` | `np.linalg.solve(A,b)` |
| Pseudoinversa | Generaliza para no-cuadradas | `np.linalg.pinv(A)` |
| Número de condición | Sensibilidad numérica | `np.linalg.cond(A)` |
| Gauss-Jordan | Aumentar $[A|I]$ hasta $[I|A^{-1}]$ | Implementación manual |

**Siguiente:** `07_vector_spaces.ipynb`